# Speech Recognition for Turkic Languages (Omnilingual ASR)

TurkicNLP's `SpeechRecognizer` turns speech into text for **20 Turkic languages**. It wraps Meta's
[Omnilingual ASR](https://github.com/facebookresearch/omnilingual-asr) models
(default: [`omniASR_LLM_1B`](https://huggingface.co/facebook/omniASR-LLM-1B)) and adds the TurkicNLP conventions:

| What | How |
|---|---|
| Language codes | TurkicNLP ISO 639-3 codes (`kaz`, `uzb`, …) are mapped to Omnilingual codes (`kaz_Cyrl`, `uzn_Latn`, …) |
| Language check | only the Turkic languages covered by TurkicNLP are accepted; each code is checked against the model's `supported_langs` |
| Output script | transcripts come in the language's primary TurkicNLP script, or one you request (via the TurkicNLP transliterator) |
| Long audio | recordings longer than the model's 40 s limit are split at pauses and joined |
| Memory | the model is loaded memory-mapped in bfloat16 (≈ 6 GB RAM for `omniASR_LLM_1B`) and shared by all languages |
| Pipeline | `Pipeline(...).from_audio()` runs speech → text → tokenisation, POS, lemmas, parsing |

**Contents**

1. Setup and environment check
2. Supported languages and language checks
3. Available models
4. Loading the recognizer (device, dtype, low-memory loading)
5. Transcribing files (single, batch, per-file languages)
6. Detailed results: segments, model language, model text
7. Output scripts and transliteration
8. Input types: paths, bytes, arrays, dicts, Hugging Face datasets
9. Long recordings
10. Explicit Omnilingual codes and unconditioned decoding
11. Speech → text → analysis with `Pipeline.from_audio`
12. Combining ASR with other TurkicNLP components
13. Switching models and the command-line script

## 1. Setup

Speech recognition needs the **speech + text environment** (Python 3.10–3.12). `omnilingual-asr` pins
`torch==2.8`, `numpy<2` and `huggingface_hub<1`, so install it in a **separate** virtual environment rather than
adding it to an existing text-only one:

```bash
# macOS: brew install libsndfile cmake      Ubuntu: sudo apt-get install libsndfile1 cmake build-essential
python3.10 -m venv venv-asr
source venv-asr/bin/activate
pip install "turkicnlp[all,asr]" jupyter ipykernel ipywidgets
python -m ipykernel install --user --name venv-asr --display-name "TurkicNLP (speech)"
```

Then select the **TurkicNLP (speech)** kernel for this notebook.

> **macOS:** if building `kenlm` fails with `'climits' file not found`, run
> `export CXXFLAGS="-nostdinc++ -isystem $(xcrun --show-sdk-path)/usr/include/c++/v1" CMAKE_POLICY_VERSION_MINIMUM=3.5`
> before `pip install` (see the TurkicNLP README, *Version constraints and known issues*).

The model weights are downloaded on first use (9.1 GB for `omniASR_LLM_1B`) and cached in `~/.cache/fairseq2/assets/`.

**Runtime.** On a laptop CPU, loading takes about 40 s and each short recording a few seconds. Running the whole
notebook takes 15–25 minutes on CPU (it also loads Stanza, Glot500 and NLLB in sections 11–12); with a GPU set
`DEVICE = "cuda"`. Sections 1–10 alone are enough to try speech recognition.

In [ ]:
# Install (run once in the speech environment)
# !pip install "turkicnlp[all,asr]"

In [ ]:
import sys, time
from importlib.metadata import version

import numpy as np
import soundfile as sf
import turkicnlp
from turkicnlp import SpeechRecognizer, list_asr_languages

print("Python          ", sys.version.split()[0])
for pkg in ["turkicnlp", "omnilingual-asr", "fairseq2", "torch", "torchaudio", "numpy", "transformers", "stanza"]:
    try:
        print(f"{pkg:<16}", version(pkg))
    except Exception:
        print(f"{pkg:<16}", "not installed")

### Sample recordings

The folder `data/asr/` contains two short test recordings (16 kHz, mono, synthesised with the macOS voices *Aru* and *Yelda*):

| File | Language | Spoken text |
|---|---|---|
| `kaz_sample.wav` | Kazakh | Мен бүгін мектепке бардым. Алматы — Қазақстанның ең үлкен қаласы. |
| `tur_sample.wav` | Turkish | Bugün hava çok güzel. Ali dün Ankara'ya gitti. |

Replace them with your own recordings (wav, flac, mp3, ogg, …) to try other languages.

In [ ]:
KAZ = "data/asr/kaz_sample.wav"
TUR = "data/asr/tur_sample.wav"

for path in [KAZ, TUR]:
    info = sf.info(path)
    print(f"{path}: {info.duration:.2f} s, {info.samplerate} Hz, {info.channels} channel(s)")

# Listen to the recordings in the notebook
from IPython.display import Audio, display
display(Audio(KAZ), Audio(TUR))

## 2. Supported languages and language checks

`list_asr_languages()` shows the 20 Turkic languages with speech recognition and the Omnilingual code used for
each script. With `check_model=True` the codes are checked against the installed model's `supported_langs`.

In [ ]:
langs = list_asr_languages(check_model=True)
print(f"{len(langs)} languages\n")
print(f"{'Code':<6} {'Script → Omnilingual code'}")
print("-" * 50)
for lang, scripts in langs.items():
    print(f"{lang:<6} " + ", ".join(f"{scr} → {c}" for scr, c in scripts.items()))

In [ ]:
# The full Omnilingual list has 1,600+ languages; TurkicNLP uses only the Turkic ones
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
print("Languages in the model:", len(supported_langs))
print("Turkic codes used by TurkicNLP:",
      sorted({c for scripts in langs.values() for c in scripts.values()}))

`resolve_asr_language` shows how a TurkicNLP code is mapped. Languages outside TurkicNLP (e.g. English, Russian)
and Turkic languages without an Omnilingual code (Tuvan, Khalaj, Ottoman Turkish, Old Turkic) are rejected.

In [ ]:
from turkicnlp.asr import resolve_asr_language

for code_ in ["kaz", "uzb", "azb", "kaa", "crh", "uig"]:
    lang, model_code, model_script = resolve_asr_language(code_)
    print(f"{code_:<4} → model code {model_code:<9} (model writes {model_script})")

print()
for code_ in ["eng", "rus_Cyrl", "tyv", "ota"]:
    try:
        resolve_asr_language(code_)
    except ValueError as err:
        print(f"{code_:<9} rejected: {err}")

## 3. Available models

Any Omnilingual ASR model card can be used. LLM models condition on the language code; CTC models are faster but
ignore it.

| Model card | Checkpoint | Notes |
|---|---|---|
| `omniASR_LLM_300M` | 6.5 GB | smallest LLM model |
| `omniASR_LLM_1B` (default) | 9.1 GB | ≈ 6 GB RAM with low-memory loading (runs on a 16 GB laptop) |
| `omniASR_LLM_3B` | 17.5 GB | GPU recommended |
| `omniASR_LLM_7B` | 31.2 GB | best quality; GPU required in practice |
| `omniASR_CTC_300M` | 1.3 GB | fast, no language conditioning |
| `omniASR_CTC_1B` | 3.9 GB | fast, no language conditioning |
| `omniASR_CTC_3B` | 12.3 GB | fast, no language conditioning |

Choose the model in the next cell. Section 13 shows how to switch models.

In [ ]:
MODEL_CARD = "omniASR_LLM_1B"   # e.g. "omniASR_LLM_300M" or "omniASR_CTC_300M" on smaller machines
DEVICE = "cpu"                  # "cuda" on an NVIDIA GPU, "mps" on Apple silicon (experimental), None = auto

## 4. Loading the recognizer

| Argument | Default | Meaning |
|---|---|---|
| `lang` | `None` | default language for all calls (can be overridden per call) |
| `model_card` | `"omniASR_LLM_1B"` | Omnilingual model |
| `script` | language's primary script | output script (`"Latn"`, `"Cyrl"`, `"Arab"`) |
| `device` | `None` → `"cuda"` if available, else `"cpu"` | where the model runs |
| `dtype` | `"auto"` → bfloat16 (float16 on `mps`) | `"float32"` doubles the memory |
| `batch_size` | `2` | segments per forward pass |
| `max_segment_seconds` | `30` | long audio is split into pieces of at most this length (≤ 40) |
| `low_memory` | `True` | memory-map the checkpoint while loading instead of reading it into RAM |
| `check_model_langs` | `True` | check codes against the model's `supported_langs` |

The model is loaded on the first call to `transcribe()`, or explicitly with `load()`. All recognizers with the same
model, device and dtype share one loaded model.

In [ ]:
asr = SpeechRecognizer(
    model_card=MODEL_CARD,
    device=DEVICE,
    dtype="bfloat16",
    low_memory=True,
    batch_size=2,
)

start = time.time()
asr.load()                     # downloads the weights on the first run
print(f"Loaded in {time.time() - start:.0f} s:", asr)

## 5. Transcribing files

The call mirrors Omnilingual's `ASRInferencePipeline.transcribe(audio_files, lang=[...], batch_size=...)`,
but takes TurkicNLP language codes. One input returns a string, a list returns a list.

In [ ]:
# One file, one language
print(asr.transcribe(KAZ, lang="kaz"))

In [ ]:
# Several files with one language code per file
texts = asr.transcribe([KAZ, TUR], lang=["kaz", "tur"], batch_size=2)
for path, text in zip([KAZ, TUR], texts):
    print(f"{path}: {text}")

In [ ]:
# A default language set on the recognizer is used when `lang` is omitted
asr_kaz = SpeechRecognizer(lang="kaz", model_card=MODEL_CARD, device=DEVICE)   # shares the loaded model
print(asr_kaz.transcribe(KAZ))

## 6. Detailed results

With `return_details=True` each input returns an `ASRResult`:

| Field | Meaning |
|---|---|
| `text` | transcript in the output script |
| `lang`, `script` | TurkicNLP language and output script |
| `model_lang` | Omnilingual code used for decoding |
| `model_text` | transcript exactly as produced by the model |
| `segments` | `ASRSegment(start, end, text)` per audio piece (seconds) |

In [ ]:
res = asr.transcribe(TUR, lang="tur", return_details=True)
print("lang:      ", res.lang)
print("script:    ", res.script)
print("model_lang:", res.model_lang)
print("text:      ", res.text)
for seg in res.segments:
    print(f"  [{seg.start:6.2f} – {seg.end:6.2f}] {seg.text}")

Omnilingual ASR writes lowercase text without punctuation. Keep this in mind when the transcript is processed further
(section 11).

## 7. Output scripts and transliteration

By default the transcript is in the language's primary TurkicNLP script. Use `script=` to request another one; the
TurkicNLP transliterator converts the model output. This also covers languages the model only writes in one script:

- **Karakalpak** and **Crimean Tatar** are written by the model in Cyrillic but are returned in their primary **Latin** script;
- **Kazakh** can be returned in the 2021 Latin alphabet, **Uzbek** in Cyrillic, **Tatar** in Zamanälif, etc.

In [ ]:
print("Cyrillic (default):", asr.transcribe(KAZ, lang="kaz"))
print("Latin (2021):      ", asr.transcribe(KAZ, lang="kaz", script="Latn"))

res = asr.transcribe(KAZ, lang="kaz", script="Latn", return_details=True)
print("\nmodel_text:", res.model_text)
print("text:      ", res.text)

In [ ]:
# Script plan for languages written differently by the model and by TurkicNLP
from turkicnlp.scripts import get_script_config

for code_ in ["kaa", "crh", "uzb", "tat", "uig"]:
    _, model_code, model_script = resolve_asr_language(code_)
    primary = get_script_config(code_).primary
    note = "" if str(primary) == model_script else f"  → transliterated to {primary}"
    print(f"{code_}: model {model_code}, TurkicNLP primary script {primary}{note}")

# With a Karakalpak recording:
# res = asr.transcribe("kaa_recording.wav", lang="kaa", return_details=True)
# res.model_text  -> Cyrillic (model output)
# res.text        -> Latin (2016 Karakalpak alphabet)

## 8. Input types

Besides file paths, `transcribe()` accepts encoded bytes, waveform arrays (with `sample_rate`), and dicts with
`waveform`/`array` and `sample_rate`/`sampling_rate` keys, which is the format of Hugging Face `datasets` audio columns.
Stereo audio is converted to mono and everything is resampled to 16 kHz.

In [ ]:
from pathlib import Path

wav, sr = sf.read(TUR)                                                     # numpy array + sample rate

print("Path      :", asr.transcribe(Path(TUR), lang="tur"))
print("Bytes     :", asr.transcribe(Path(TUR).read_bytes(), lang="tur"))
print("Array     :", asr.transcribe(wav, lang="tur", sample_rate=sr))
print("Dict      :", asr.transcribe({"waveform": wav, "sample_rate": sr}, lang="tur"))
print("HF-style  :", asr.transcribe({"array": wav, "sampling_rate": sr}, lang="tur"))

stereo = np.stack([wav, wav], axis=1)                                      # (frames, 2)
print("Stereo    :", asr.transcribe(stereo, lang="tur", sample_rate=sr))

In [ ]:
# Hugging Face datasets (install `datasets` first); audio columns can be passed directly:
#
# from datasets import load_dataset, Audio as HFAudio
# ds = load_dataset("<a Turkic speech dataset>", split="test").cast_column("audio", HFAudio(sampling_rate=16000))
# hyps = asr.transcribe(list(ds["audio"][:8]), lang="kir", batch_size=4)
# refs = ds["sentence"][:8]

## 9. Long recordings

Omnilingual ASR accepts at most 40 s per input. `SpeechRecognizer` splits longer audio into pieces of at most
`max_segment_seconds` (default 30 s), cutting at the quietest point near each boundary, and joins the transcripts.
Here we build a ~17 s recording (three copies of the Kazakh sample with pauses) and use 8-second pieces so the
splitting is visible without a long run.

In [ ]:
from turkicnlp.asr import split_audio

kaz_wav, sr = sf.read(KAZ)
pause = np.zeros(int(1.0 * sr))
long_wav = np.concatenate([kaz_wav, pause] * 3)
print(f"Duration: {len(long_wav) / sr:.1f} s")

# Where would it be cut?
for s, e in split_audio(long_wav, sr, max_seconds=8):
    print(f"  piece {s / sr:6.2f} – {e / sr:6.2f} s")

In [ ]:
asr_long = SpeechRecognizer(model_card=MODEL_CARD, device=DEVICE, max_segment_seconds=8)  # shares the model
res = asr_long.transcribe(long_wav, lang="kaz", sample_rate=sr, return_details=True)
for seg in res.segments:
    print(f"[{seg.start:6.2f} – {seg.end:6.2f}] {seg.text}")
print("\nFull text:", res.text)

## 10. Explicit Omnilingual codes and unconditioned decoding

- An Omnilingual code of a covered Turkic language (e.g. `"uzb_Cyrl"`, `"uig_Cyrl"`) selects that code directly;
  the output stays in the code's script.
- `lang=None` decodes without a language token. This works, but LLM models are less accurate without it.

In [ ]:
print("Explicit code kaz_Cyrl:", asr.transcribe(KAZ, lang="kaz_Cyrl"))
print("No language token:     ", SpeechRecognizer(model_card=MODEL_CARD, device=DEVICE).transcribe(KAZ))

## 11. Speech → text → analysis

`Pipeline(...).from_audio()` transcribes the audio and runs the text processors on the transcript. Options for the
recognizer are passed with the `asr_` prefix: `asr_model_card`, `asr_device`, `asr_dtype`, `asr_batch_size`,
`asr_max_segment_seconds`. The document gets `audio_segments` with the time-aligned transcript.

In [ ]:
nlp = turkicnlp.Pipeline(
    "kaz",
    processors=["asr", "tokenize", "pos", "lemma", "depparse"],
    asr_model_card=MODEL_CARD,
    asr_device=DEVICE,
)

doc = nlp.from_audio(KAZ)
print("Text:    ", doc.text)
print("Segments:", doc.audio_segments)
print()
print(f"{'Word':<16} {'UPOS':<6} {'Lemma':<14} {'Head':>4}  Deprel")
for w in doc.words:
    print(f"{w.text:<16} {w.upos or '':<6} {w.lemma or '':<14} {w.head!s:>4}  {w.deprel}")

In [ ]:
# Several recordings at once, and CoNLL-U export
docs = nlp.from_audio([KAZ, KAZ])
print(len(docs), "documents")
print(docs[0].to_conllu())

In [ ]:
# Multilingual Glot500 backends work the same way
nlp_glot = turkicnlp.Pipeline(
    "tur",
    processors=["asr", "tokenize", "pos", "depparse"],
    pos_backend="multilingual_glot500_model",
    depparse_backend="multilingual_glot500_model",
    asr_model_card=MODEL_CARD, asr_device=DEVICE,
)
doc = nlp_glot.from_audio(TUR)
print([(w.text, w.upos, w.deprel) for w in doc.words])

## 12. Combining ASR with other TurkicNLP components

Transcripts are ordinary TurkicNLP text, so the rest of the toolkit applies directly: language identification,
transliteration to the Common Turkic Alphabet, morpheme segmentation, and translation.

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

kaz_text = asr.transcribe(KAZ, lang="kaz")
tur_text = asr.transcribe(TUR, lang="tur")

# Language identification on the transcripts (GlotLID, Turkic labels)
lid = turkicnlp.LanguageDetection()
for text in [kaz_text, tur_text]:
    labels, probs = lid.predict(text, k=1)
    print(f"{labels[0]:<20} {probs[0]:.2f}  {text}")

# Common Turkic Alphabet: one Latin orthography for both transcripts
print()
print("kaz → CTS:", Transliterator("kaz", Script.CYRILLIC, Script.COMMON_TURKIC).transliterate(kaz_text))
print("tur → CTS:", Transliterator("tur", Script.LATIN, Script.COMMON_TURKIC).transliterate(tur_text))

In [ ]:
# Morpheme segmentation of the transcribed words
from turkicnlp.processors.morpheme_tokenizer import MorphemeTokenizer

seg = MorphemeTokenizer(lang="kaz"); seg.load()
for word in kaz_text.split()[:5]:
    print(f"{word:<16}", seg.segment(word).labeled)

In [ ]:
# Speech translation: transcribe, then translate with NLLB-200 (non-commercial license)
mt = turkicnlp.Pipeline("kaz", processors=["translate"], translate_tgt_lang="eng")
print("kaz:", kaz_text)
print("eng:", mt(kaz_text).translation)

## 13. Switching models and the command-line script

Changing the model only changes `model_card`; the API stays the same. Larger models need more memory (section 3).
Set `RUN_OTHER_MODEL = True` to try a second model (this downloads its weights).

In [ ]:
RUN_OTHER_MODEL = False
OTHER_MODEL = "omniASR_CTC_300M"       # 1.3 GB; CTC models ignore the language code

if RUN_OTHER_MODEL:
    asr_other = SpeechRecognizer(model_card=OTHER_MODEL, device=DEVICE)
    start = time.time()
    print(asr_other.transcribe([KAZ, TUR], lang=["kaz", "tur"]))
    print(f"{OTHER_MODEL}: {time.time() - start:.0f} s (including loading)")

**Memory tips**

- Keep `low_memory=True` and `dtype="bfloat16"` (the defaults). Without them, `omniASR_LLM_1B` needs more than 16 GB.
- Use `omniASR_LLM_300M` or a CTC model on machines with less memory.
- `device="cuda"` uses an NVIDIA GPU; larger models (3B, 7B) are practical only on a GPU.

**Command line.** The TurkicNLP repository includes `run_asr.py`:

```bash
python run_asr.py data/asr/kaz_sample.wav data/asr/tur_sample.wav --lang kaz tur --device cpu
python run_asr.py lecture.wav --lang tat --model omniASR_LLM_300M
python run_asr.py interview.wav --lang uzb --script Cyrl --analyze      # + tokenize/POS/depparse
```

Options: `--model`, `--device`, `--dtype`, `--script`, `--batch-size`, `--no-low-memory`, `--analyze`.

## Summary

| Task | Code |
|---|---|
| Transcribe | `SpeechRecognizer().transcribe(path, lang="kaz")` |
| Batch with per-file languages | `asr.transcribe([a, b], lang=["kaz", "tur"], batch_size=2)` |
| Segments and model output | `asr.transcribe(path, lang="kaz", return_details=True)` |
| Other script | `asr.transcribe(path, lang="kaz", script="Latn")` |
| Supported languages | `turkicnlp.list_asr_languages(check_model=True)` |
| Speech → analysis | `Pipeline("kaz", processors=["asr", "tokenize", "pos"]).from_audio(path)` |
| Smaller / faster model | `SpeechRecognizer(model_card="omniASR_LLM_300M")` |

**References**

- Omnilingual ASR: [github.com/facebookresearch/omnilingual-asr](https://github.com/facebookresearch/omnilingual-asr), [huggingface.co/facebook/omniASR-LLM-1B](https://huggingface.co/facebook/omniASR-LLM-1B)
- TurkicNLP: [turkic-nlp.github.io](https://turkic-nlp.github.io)